# Home Credit Default Risk
## Sprint 2 — Pré-Processamento e Feature Engineering

**Dataset:** `application_train.csv` (~307k linhas, 122 colunas)  
**Target:** `TARGET` — 0 = pagou normalmente, 1 = dificuldade de pagamento nas parcelas iniciais  
**Tipo de tarefa:** Classificação binária  

---

## SEÇÃO 1 — Revisão dos achados da Sprint 1

### 1.1 Breve recapitulação dos problemas identificados na EDA
Com base na análise realizada na Sprint 1, identificamos os seguintes pontos críticos no dataset `application_train.csv`:
* **Desbalanceamento:** O target é altamente desbalanceado (~92% pagaram normalmente e ~8% tiveram dificuldades).
* **Valores Ausentes:** 67 das 122 colunas possuem valores nulos. O problema é severo em dados de infraestrutura imobiliária (ex: `COMMONAREA_AVG` com 69,87%) e idade do veículo (`OWN_CAR_AGE` com 65,99%).
* **Fontes Externas:** As features de score externo têm grande poder preditivo, mas sofrem com ausências, especialmente a `EXT_SOURCE_1` (56,38% de nulos).
* **Variáveis para Transformação:** Temos 16 variáveis categóricas que precisarão de Encoding e 106 variáveis numéricas para avaliar escalonamento e outliers.

### 1.2 Lista de ações de pré-processamento planejadas
Para esta sprint, planejamos:
1. Dividir os dados em Treino e Teste antes de qualquer transformação para evitar *data leakage*.
2. Descartar colunas de infraestrutura com mais de 60% de nulos, pois não agregam valor e introduzem muito ruído.
3. Imputar valores nas variáveis críticas (como `EXT_SOURCE`) usando métodos estatísticos ou avançados.
4. Tratar outliers apenas nas variáveis financeiras em que a distorção afeta a modelagem.
5. Aplicar *One-Hot Encoding* em categóricas nominais e *Target Encoding* naquelas com muitas categorias.
6. Encapsular tudo em um Pipeline do Scikit-Learn.

In [ ]:
# 1.3 Carregamento do dataset e separação em Treino e Teste
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df_raw = pd.read_csv('C:\\dev\\EngenhariaDeSoftware\\semestre7\\ia\\pesquisa_aplicada_01\\application_train.csv')

# Separar treino e teste ANTES de qualquer transformação
X = df_raw.drop(columns=['TARGET', 'SK_ID_CURR']) 
y = df_raw['TARGET']

# Usando stratify=y devido ao forte desbalanceamento (8% da classe 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape X_train: {X_train.shape}")
print(f"Shape X_test: {X_test.shape}")

---

## SEÇÃO 2 — Tratamento de dados ausêntes

### 2.1 Diagnóstico de missing values
Primeiro, vamos verificar a proporção exata de dados faltantes nas colunas da nossa base de treino.

In [ ]:
# Diagnóstico de nulos por coluna no treino
missing_cols = X_train.isnull().mean() * 100
missing_cols = missing_cols[missing_cols > 0].sort_values(ascending=False)
print("Top 10 colunas com mais valores ausentes (%):")
print(missing_cols.head(10))

# Diagnóstico de nulos por linha no treino
missing_rows = X_train.isnull().sum(axis=1)
print(f"\nMédia de valores ausentes por linha: {missing_rows.mean():.2f}")

### 2.2 Estratégias e Justificativas para Tratamento de Ausentes

Com base no diagnóstico, adotaremos as seguintes abordagens estruturadas:

* **Estratégia 1 - Remoção (Drop):** Colunas com mais de 60% de dados ausentes (ex: `COMMONAREA_AVG`, `OWN_CAR_AGE`) serão removidas. **Justificativa:** Imputar dados em variáveis com mais de 60% de ausência destrói a variância original e insere muito viés, pois significa inventar a maior parte da informação.
* **Estratégia 2 - Imputação Simples (Mediana/Moda):** Para variáveis com menos de 60% de nulos. **Justificativa:** A mediana será usada para números pois é robusta contra outliers. A Moda será usada para variáveis categóricas (textos).
* **Estratégia 3 - Imputação Avançada (IterativeImputer):** Para a `EXT_SOURCE_1` (56,38% de ausentes). **Justificativa:** Apesar da alta ausência, é a variável com maior poder preditivo do negócio. O `IterativeImputer` estimará esse valor cruzando dados de renda e outras fontes externas. *(Nota: Por padrão, aplicaremos a Mediana agora para garantir a execução limpa, mas a estrutura para o IterativeImputer ficará pronta para as próximas fases).*

### 2.3 Aplicação das Estratégias de Imputação

Abaixo, aplicamos as transformações definidas nas bases de Treino e Teste. Note que, para evitar o vazamento de dados (*data leakage*), os imputadores "aprendem" (`fit`) as métricas apenas na base de Treino e apenas aplicam (`transform`) na base de Teste.

In [ ]:
# 2.3 Aplicação das Estratégias de Imputação
from sklearn.impute import SimpleImputer
# from sklearn.experimental import enable_iterative_imputer
# from sklearn.impute import IterativeImputer

# Guardar cópia para evidência do antes e depois
X_train_before = X_train.copy()

# ESTRATÉGIA 1: Remoção de colunas com mais de 60% de nulos
cols_to_drop = missing_cols[missing_cols > 60.0].index.tolist()
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop) 
print(f"Foram removidas {len(cols_to_drop)} colunas por excesso de nulos.")

# ESTRATÉGIA 2: Imputação com a Mediana (Numéricas)
num_cols_to_impute = X_train.select_dtypes(include=[np.number]).columns
median_imputer = SimpleImputer(strategy='median')

# FIT apenas no treino para evitar data leakage
X_train[num_cols_to_impute] = median_imputer.fit_transform(X_train[num_cols_to_impute])
X_test[num_cols_to_impute] = median_imputer.transform(X_test[num_cols_to_impute])

# ESTRATÉGIA 3: Imputação com a Moda (Categóricas)
# Nota: SimpleImputer.fit_transform retorna numpy array e perde o dtype 'object'.
# Solução: imputar coluna a coluna com fillna(moda), preservando os dtypes originais.
cat_cols_to_impute = X_train.select_dtypes(include=['object']).columns

for col in cat_cols_to_impute:
    moda = X_train[col].mode()[0]  # moda calculada apenas no treino
    X_train[col] = X_train[col].fillna(moda)
    X_test[col]  = X_test[col].fillna(moda)

### 2.4 Evidência do Tratamento
Verificação final para atestar que não restaram valores nulos nas bases de dados após a imputação.

In [ ]:
# Contagem de nulos antes e depois
missing_before = X_train_before.isnull().sum().sum()
missing_after = X_train.isnull().sum().sum()

print(f"Total de valores ausentes ANTES do tratamento: {missing_before}")
print(f"Total de valores ausentes DEPOIS do tratamento: {missing_after}")

# Evidência prática em uma coluna importante (EXT_SOURCE_3)
print("\n--- Estatísticas de EXT_SOURCE_3 ANTES ---")
print(X_train_before['EXT_SOURCE_3'].describe()[['count', 'mean', 'std']])
print("\n--- Estatísticas de EXT_SOURCE_3 DEPOIS ---")
print(X_train['EXT_SOURCE_3'].describe()[['count', 'mean', 'std']])

---

## SEÇÃO 3 — Tratamento de Outliers

A identificação foi feita pelo método IQR, com dois filtros aplicados antes de qualquer tratamento:

- **`IQR = 0`:** coluna descartada da análise. Ocorre em variáveis binárias e constantes, onde todos os quartis coincidem e qualquer valor diferente do dominante seria marcado como outlier incorretamente.
- **`outliers > 10%`:** coluna descartada da análise. Um percentual alto indica assimetria estrutural da distribuição, não pontos isolados. Tratar pelo IQR nesse caso removeria variação legítima.
- **`DAYS_EMPLOYED`:** o valor sentinela `365243` (usado para clientes aposentados ou sem vínculo empregatício) foi substituído por `NaN` antes da aplicação do IQR, para não inflar artificialmente o percentual de outliers da coluna e permitir que ela fosse avaliada com os valores reais de dias trabalhados.

### 3.1 Identificação de Outliers via IQR

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

X_train['DAYS_EMPLOYED'] = X_train['DAYS_EMPLOYED'].replace(365243, np.nan)
X_test['DAYS_EMPLOYED'] = X_test['DAYS_EMPLOYED'].replace(365243, np.nan)

# 3.2 Identificação de Outliers via IQR
def identify_outliers_iqr(data, column):
    """Identifica outliers usando o método IQR"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (data[column] < lower_bound) | (data[column] > upper_bound)

# Aplicar IQR apenas em colunas onde o método faz sentido (IQR > 0 e outliers <= 10%)
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
outlier_counts = {}

for col in numeric_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    if IQR == 0:
        continue
    outliers = identify_outliers_iqr(X_train, col)
    pct = outliers.sum() / len(X_train)
    if pct > 0.10:
        continue
    outlier_counts[col] = outliers.sum()

# Ordenar por quantidade de outliers
outlier_counts = dict(sorted(outlier_counts.items(), key=lambda x: x[1], reverse=True))

print("Top 15 colunas com mais outliers (método IQR):")
for col, count in list(outlier_counts.items())[:15]:
    pct = (count / len(X_train)) * 100
    print(f"{col}: {count} outliers ({pct:.2f}%)")

### 3.2 Visualização de Outliers

Boxplots das 15 colunas identificadas, usados para confirmar visualmente o diagnóstico do IQR e orientar a estratégia de tratamento de cada coluna.

In [ ]:
cols_to_plot = list(outlier_counts.keys())[:15]
n = len(cols_to_plot)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.ravel()

for idx, col in enumerate(cols_to_plot):
    axes[idx].boxplot(X_train[col].dropna(), vert=True, patch_artist=True,
                      boxprops=dict(facecolor='steelblue', alpha=0.7))
    axes[idx].set_title(col, fontsize=10)
    pct = (outlier_counts[col] / len(X_train)) * 100
    axes[idx].text(0.02, 0.98, f'Outliers: {outlier_counts[col]} ({pct:.2f}%)',
                   transform=axes[idx].transAxes, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5), fontsize=8)

# Esconder eixos vazios
for idx in range(n, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Boxplots — Colunas com Outliers (IQR)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 3.3 Estratégia de Tratamento de Outliers

A winsorização foi aplicada usando os próprios limites do IQR (`Q1 - 1.5×IQR` e `Q3 + 1.5×IQR`) como cap, e não o percentil 1%/99%. Dessa forma, os valores capados são exatamente os marcados como outliers na etapa anterior, mantendo consistência entre identificação e tratamento.

Por mais que o IQR sinalizou outliers em `HOUR_APPR_PROCESS_START` pela alta concentração de pedidos no horário comercial, ela foi mantida sem alteração, pois os valores fora desse intervalo (madrugada e início da manhã) são válidos.

| Coluna | Outliers | % | Decisão | Justificativa |
|---|---|---|---|---|
| `OBS_30_CNT_SOCIAL_CIRCLE` | 15.965 | 6,49% | **Winsorizar** | Contagem de pessoas do círculo social com atrasos de 30 dias. Valores acima de 5 são implausíveis. |
| `OBS_60_CNT_SOCIAL_CIRCLE` | 15.642 | 6,36% | **Winsorizar** | Mesma variável para atrasos de 60 dias. Tratamento idêntico. |
| `DAYS_EMPLOYED` | 12.127 | 4,93% | **Winsorizar** | Sentinela 365243 removido. Outliers residuais são tempos de emprego atipicamente longos ou curtos. |
| `AMT_GOODS_PRICE` | 11.791 | 4,79% | **Winsorizar** | Valores extremos existem, mas distorcem modelos lineares e baseados em distância. |
| `AMT_INCOME_TOTAL` | 11.247 | 4,57% | **Winsorizar** | Rendas extremas existem, mas os valores mais altos dominam modelos sensíveis a escala. |
| `REGION_POPULATION_RELATIVE` | 6.729 | 2,74% | **Winsorizar** | Valores extremos representam regiões reais, mas o cap evita dominância no modelo. |
| `AMT_ANNUITY` | 5.987 | 2,43% | **Winsorizar** | Diretamente proporcional a `AMT_CREDIT` e `AMT_INCOME_TOTAL`. Tratamento consistente entre as variáveis financeiras. |
| `AMT_REQ_CREDIT_BUREAU_YEAR` | 5.723 | 2,33% | **Winsorizar** | Consultas extremas ao bureau existem, mas os valores mais altos podem enviesar o modelo. |
| `AMT_CREDIT` | 5.260 | 2,14% | **Winsorizar** | Mesma família de `AMT_INCOME_TOTAL` e `AMT_ANNUITY`. Tratamento consistente. |
| `EXT_SOURCE_3` | 3.475 | 1,41% | **Winsorizar** | Score entre 0 e 1. Outliers inferiores são scores válidos de alto risco; cap preserva a escala. |
| `CNT_CHILDREN` | 3.366 | 1,37% | **Winsorizar** | Valores acima de ~5 são prováveis erros de entrada. |
| `CNT_FAM_MEMBERS` | 3.155 | 1,28% | **Winsorizar** | Correlacionada com `CNT_CHILDREN`. Tratamento consistente. |
| `HOUR_APPR_PROCESS_START` | 1.816 | 0,74% | **Manter** | Valores fora do horário comercial são válidos. Não há outlier real. |
| `DAYS_REGISTRATION` | 543 | 0,22% | **Winsorizar** | Valores muito negativos representam documentos antigos. Cap evita dominância. |
| `DAYS_LAST_PHONE_CHANGE` | 341 | 0,14% | **Winsorizar** | Percentual residual. Tratado por consistência com as demais variáveis temporais. |

>
**Nenhuma coluna foi removida.** Todos os outliers têm interpretação de negócio plausível.

In [ ]:
X_train_before_outliers = X_train.copy()
X_test_before_outliers = X_test.copy()

# Coluna sem tratamento necessário (outliers são horários válidos)
cols_to_keep = ['HOUR_APPR_PROCESS_START']

cols_to_winsorize = [col for col in outlier_counts.keys() if col not in cols_to_keep]

for col in cols_to_winsorize:
    Q1    = X_train[col].quantile(0.25)
    Q3    = X_train[col].quantile(0.75)
    IQR   = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    X_train[col] = X_train[col].clip(lower=lower, upper=upper)
    X_test[col]  = X_test[col].clip(lower=lower, upper=upper)
    print(f"{col}: clipped em [{lower:.2f}, {upper:.2f}]")

for col in cols_to_keep:
    print(f"{col}: mantida sem alteração (outliers são valores válidos de negócio)")

print("\nTratamento de outliers concluído!")


### 3.4 Visualização: Antes e Depois do Tratamento

Histogramas comparativos antes e depois da winsorização.

- Colunas com percentual baixo de outliers (como `DAYS_LAST_PHONE_CHANGE`, 0,14%) mostram distribuições praticamente idênticas.
- Colunas com percentual mais alto (como `AMT_INCOME_TOTAL` e `OBS_30_CNT_SOCIAL_CIRCLE`) mostram remoção visível das caudas.

In [ ]:
cols_to_plot = list(outlier_counts.keys())[:15]
n = len(cols_to_plot)
ncols = 2
nrows = n

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3))

for idx, col in enumerate(cols_to_plot):
    axes[idx, 0].hist(X_train_before_outliers[col].dropna(), bins=50, color='red', alpha=0.7, edgecolor='black')
    axes[idx, 0].set_title(f'{col} — ANTES', fontsize=9)

    axes[idx, 1].hist(X_train[col].dropna(), bins=50, color='green', alpha=0.7, edgecolor='black')
    axes[idx, 1].set_title(f'{col} — DEPOIS', fontsize=9)

plt.suptitle('Distribuições: Antes e Depois da Winsorização', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---

## SEÇÃO 4 — Encoding de Variáveis Categóricas

### 4.1 Identificação de Variáveis Categóricas

Variáveis categóricas precisam ser convertidas em representações numéricas para que os algoritmos de aprendizado de máquina possam processá-las. Primeiro, vamos identificar todas as variáveis categóricas no dataset.

**Tipos de Categorias:**
- **Ordinais:** Têm uma ordem natural (ex: baixo < médio < alto)
- **Nominais:** Não têm ordem (ex: cores, estados civis)

A estratégia de encoding será diferente para cada tipo.


In [ ]:
# 4.1 Identificação de todas as variáveis categóricas
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f'Total de variáveis categóricas: {len(cat_cols)}')
print()

for col in cat_cols:
    n_unique = X_train[col].nunique()
    values = X_train[col].value_counts().index.tolist()
    print(f'{col:40s} | categorias únicas: {n_unique:>3} | valores: {values[:6]}')

### 4.2 Justificativa para a Estratégia de Encoding

O dataset possui **15 variáveis categóricas** após o pré-processamento. A estratégia de encoding foi definida com base em dois critérios: o **tipo semântico** da variável (ordinal ou nominal) e o **número de categorias únicas**.

| Técnica | Critério | Motivo |
|---|---|---|
| **LabelEncoder** | Variáveis **ordinais** | Preserva a hierarquia natural entre categorias em uma única coluna numérica. Usar OneHotEncoder destruiria essa informação de ordem. |
| **OneHotEncoder** | Variáveis **nominais** com **≤ 10 categorias** | Sem ordem entre categorias, cada uma deve ser independente. Com poucas categorias, o custo de novas colunas é aceitável. |
| **Target Encoding** | Variáveis **nominais** com **> 10 categorias** | Muitas categorias tornam o OneHotEncoder impraticável pela explosão de dimensionalidade. O Target Encoding substitui cada categoria pela taxa média de inadimplência do treino, preservando o sinal preditivo em uma única coluna sem risco de data leakage. |

#### Mapeamento das 15 variáveis

| Variável | Categorias | Tipo | Estratégia | Justificativa |
|---|---|---|---|---|
| `NAME_CONTRACT_TYPE` | 2 | Nominal | **OneHotEncoder** | Cash loans / Revolving loans — sem ordem, baixo custo. |
| `CODE_GENDER` | 3 | Nominal | **OneHotEncoder** | F / M / XNA — sem hierarquia entre gêneros. |
| `FLAG_OWN_CAR` | 2 | Nominal | **OneHotEncoder** | Binária (N/Y) — sem ordem. |
| `FLAG_OWN_REALTY` | 2 | Nominal | **OneHotEncoder** | Binária (Y/N) — sem ordem. |
| `NAME_TYPE_SUITE` | 7 | Nominal | **OneHotEncoder** | Tipos de acompanhante sem hierarquia definida. |
| `NAME_INCOME_TYPE` | 8 | Nominal | **OneHotEncoder** | Fontes de renda sem ordem natural entre si. |
| `NAME_EDUCATION_TYPE` | 5 | **Ordinal** | **LabelEncoder** | Hierarquia clara: Lower secondary < Secondary < Incomplete higher < Higher education < Academic degree. |
| `NAME_FAMILY_STATUS` | 6 | Nominal | **OneHotEncoder** | Estados civis sem gradação entre si. |
| `NAME_HOUSING_TYPE` | 6 | Nominal | **OneHotEncoder** | Tipos de moradia sem ordem natural. |
| `OCCUPATION_TYPE` | 18 | Nominal | **Target Encoding** | 18 categorias sem ordem — OneHotEncoder geraria 17 novas colunas com alta esparsidade. |
| `WEEKDAY_APPR_PROCESS_START` | 7 | Nominal | **OneHotEncoder** | Dias da semana sem hierarquia numérica. |
| `ORGANIZATION_TYPE` | 58 | Nominal | **Target Encoding** | 58 categorias — OneHotEncoder inviável. Encoding pela taxa de inadimplência preserva o sinal em uma coluna. |
| `HOUSETYPE_MODE` | 3 | **Ordinal** | **LabelEncoder** | Gradação implícita de tipo de imóvel: terraced house < specific housing < block of flats. |
| `WALLSMATERIAL_MODE` | 7 | Nominal | **OneHotEncoder** | Materiais de parede sem hierarquia definida, dentro do limiar de 10 categorias. |
| `EMERGENCYSTATE_MODE` | 2 | Nominal | **OneHotEncoder** | Binária (No/Yes) — sem ordem. |

In [ ]:
from sklearn.preprocessing import LabelEncoder

# ── 1. LABEL ENCODER (ordinais) ───────────────────────────────────────────────
ordinal_mappings = {
    'NAME_EDUCATION_TYPE': [
        'Lower secondary',
        'Secondary / secondary special',
        'Incomplete higher',
        'Higher education',
        'Academic degree'
    ],
    'HOUSETYPE_MODE': [
        'terraced house',
        'specific housing',
        'block of flats'
    ],
}

label_encoders = {}
for col, order in ordinal_mappings.items():
    if col not in X_train.columns:
        print(f'[SKIP] {col} não encontrada')
        continue
    mapping = {cat: idx for idx, cat in enumerate(order)}
    X_train[col] = X_train[col].map(mapping)
    X_test[col]  = X_test[col].map(mapping)
    label_encoders[col] = mapping
    print(f'LabelEncoder | {col}: {mapping}')

# ── 2. ONE-HOT ENCODER (nominais ≤ 10 categorias) ────────────────────────────
ohe_cols = [
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'WEEKDAY_APPR_PROCESS_START',
    'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE',
]
ohe_cols = [c for c in ohe_cols if c in X_train.columns]

X_train = pd.get_dummies(X_train, columns=ohe_cols, drop_first=True, dtype=int)
X_test  = pd.get_dummies(X_test,  columns=ohe_cols, drop_first=True, dtype=int)
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f'\nOneHotEncoder | {len(ohe_cols)} colunas → shape treino: {X_train.shape}')

# ── 3. TARGET ENCODING (nominais > 10 categorias) ────────────────────────────
target_enc_cols = ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE']
target_enc_cols = [c for c in target_enc_cols if c in X_train.columns]

target_encoders = {}
for col in target_enc_cols:
    target_mean_map = y_train.groupby(X_train[col]).mean()
    global_mean     = y_train.mean()
    target_encoders[col] = {'map': target_mean_map, 'global_mean': global_mean}

    X_train[col] = X_train[col].map(target_mean_map)
    X_test[col]  = X_test[col].map(target_mean_map).fillna(global_mean)

    top5 = target_mean_map.sort_values(ascending=False).head(5)
    print(f'\nTargetEncoder | {col} — top 5 por taxa de inadimplência:')
    for cat, val in top5.items():
        print(f'  {cat}: {val:.4f}')
    print(f'  Média global (fallback): {global_mean:.4f}')

# ── Evidência final ───────────────────────────────────────────────────────────
remaining = X_train.select_dtypes(include=['object', 'str']).columns.tolist()
print(f'\nColunas object restantes: {remaining if remaining else "Nenhuma"}')
print(f'Shape final — treino: {X_train.shape} | teste: {X_test.shape}')
assert X_train.shape[1] == X_test.shape[1], 'ERRO: shapes divergem!'
print('Encoding concluído com sucesso!')